# EDA (I)
In this EDA, I
1. explored the columns
2. figured out which are necessary
3. understood the data issues
4. tried with a single busroute for to understand travel time patterns


In [17]:
import polars as pl
import plotly.express as px
from datetime import datetime, date, timedelta
from constants import DATA_FILE, MIN_DATE, MAX_DATE
from helpers import clean_df

pl.Config.set_tbl_rows(30)

polars.config.Config

In [19]:
df_full = pl.scan_parquet(DATA_FILE)
df_full.head(5).collect().glimpse()

Rows: 5
Columns: 29
$ PlateNumb         <str> 'KKA-3975', '396-FQ', '396-FQ', '396-FQ', '396-FQ'
$ OperatorID        <i32> 16, 35, 35, 35, 35
$ OperatorNo        <i32> 702, 1308, 1308, 1308, 1308
$ RouteUID          <str> 'THB0968', 'THB0968', 'THB0968', 'THB0968', 'THB0968'
$ RouteID           <i32> 968, 968, 968, 968, 968
$ RouteNameZh_tw    <str> '0968', '0968', '0968', '0968', '0968'
$ RouteNameEn       <str> '0968', '0968', '0968', '0968', '0968'
$ SubRouteUID       <str> 'THB096802', 'THB096802', 'THB096802', 'THB096802', 'THB096802'
$ SubRouteID        <str> '096802', '096802', '096802', '096802', '096802'
$ SubRouteNameZh_tw <str> '0968', '0968', '0968', '0968', '0968'
$ SubRouteNameEn    <str> '0968', '0968', '0968', '0968', '0968'
$ Direction         <cat> 返程, 返程, 返程, 返程, 返程
$ StopUID           <str> 'THB300178', 'THB300198', 'THB300198', 'THB300197', 'THB300197'
$ StopID            <i32> 300178, 300198, 300198, 300197, 300197
$ StopNameZh_tw     <str> '大竹消防隊', '庫倫街口', '庫倫街口'

In [3]:
df_full.filter(pl.col("GPSTime") != "2").select(pl.max("GPSTime").alias("min")).collect(
    engine="streaming"
)


min
str
"""2026-02-27T23:59:58+08:00"""


In [4]:
df_full.filter(pl.col("GPSTime").str.starts_with("2025-02-28")).drop(
    [
        "GPSTime",
        "TripStartTime",
        "TransTime",
        "SrcRecTime",
        "SrcTransTime",
        "SrcUpdateTime",
        "UpdateTime",
    ]
).select(pl.all().n_unique()).collect()

PlateNumb,OperatorID,OperatorNo,RouteUID,RouteID,RouteNameZh_tw,RouteNameEn,SubRouteUID,SubRouteID,SubRouteNameZh_tw,SubRouteNameEn,Direction,StopUID,StopID,StopNameZh_tw,StopNameEn,StopSequence,MessageType,DutyStatus,BusStatus,A2EventType,TripStartTimeType
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
2331,47,47,467,467,467,467,1326,1326,701,701,2,18327,18327,6573,6955,212,1,3,5,2,3


In [5]:
# Prepare df of a specific route

TARGET_ROUTE = "17280"

In [6]:
df = (
    df_full.filter(
        pl.col("SubRouteID").str.starts_with(TARGET_ROUTE),
        pl.col("TripStartTimeType") == "實際發車時間",
    )
    .with_columns(
        pl.col("GPSTime")
        .str.replace(r"\+08:00", "")
        .str.to_datetime(format="%Y-%m-%dT%H:%M:%S")
        .alias("Time"),
        # The last digit of SubRouteID represents same info from Direction
        # Separating them would be clearer
        pl.col("SubRouteID").str.replace(r".$", "").alias("Route"),
    )
    .select(
        pl.col("Route"),
        pl.col("PlateNumb").alias("Plate"),
        pl.col("Direction"),
        pl.col("StopNameZh_tw").alias("Stop"),
        pl.col("A2EventType").alias("Event"),
        pl.col("Time"),
    )
    .filter(
        pl.col("Time").dt.date() == date(2026, 2, 25),
    )
    .collect()
)
# This was later separated into clean_df in helpers.py
df.glimpse()

Rows: 1795
Columns: 6
$ Route              <str> '17280', '17280', '17280', '17280', '17280', '17280', '17280', '17280', '17280', '17280'
$ Plate              <str> '057-FS', '057-FS', '057-FS', 'KKB-2059', '057-FS', 'KKB-2057', '057-FS', 'KKB-1713', 'KKB-2670', '057-FS'
$ Direction          <cat> 返程, 返程, 返程, 返程, 返程, 去程, 返程, 去程, 返程, 去程
$ Stop               <str> '幸安國小', '幸安國小', '仁愛建國路口一', '清華大學', '仁愛建國路口一', '捷運公館站', '捷運景美站', '龍潭運動公園', '科技生活館', '師大分部'
$ Event              <cat> 離站, 進站, 進站, 進站, 離站, 進站, 離站, 離站, 離站, 離站
$ Time      <datetime[μs]> 2026-02-25 16:15:27, 2026-02-25 16:15:14, 2026-02-25 16:15:32, 2026-02-25 16:15:38, 2026-02-25 16:15:50, 2026-02-25 16:15:59, 2026-02-25 21:21:14, 2026-02-25 21:21:34, 2026-02-25 21:21:02, 2026-02-25 11:22:13



In [7]:
df.filter(pl.col("Stop") == "新竹轉運站", pl.col("Direction") == "返程")

Route,Plate,Direction,Stop,Event,Time
str,str,cat,str,cat,datetime[μs]
"""17280""","""KKB-1692""","""返程""","""新竹轉運站""","""離站""",2026-02-25 17:51:43
"""17280""","""KKB-2669""","""返程""","""新竹轉運站""","""離站""",2026-02-25 15:00:47
"""17280""","""KKB-1713""","""返程""","""新竹轉運站""","""離站""",2026-02-25 17:21:16
"""17280""","""KKB-1692""","""返程""","""新竹轉運站""","""離站""",2026-02-25 12:01:34
"""17280""","""KKB-2057""","""返程""","""新竹轉運站""","""離站""",2026-02-25 06:20:04
"""17280""","""057-FS""","""返程""","""新竹轉運站""","""離站""",2026-02-25 14:02:23
"""17280""","""KKB-2055""","""返程""","""新竹轉運站""","""離站""",2026-02-25 08:01:17
"""17280""","""KKB-2058""","""返程""","""新竹轉運站""","""離站""",2026-02-25 05:15:46
"""17280""","""KKB-2059""","""返程""","""新竹轉運站""","""離站""",2026-02-25 09:01:20


In [8]:
fig = px.bar(
    df.filter(pl.col("Direction") == "返程")
    .group_by(["Stop", "Event"])
    .len()
    .sort(["Stop", "Event"]),
    y="Stop",
    x="len",
    color="Event",
)
fig.show()
# fig.show(renderer = "browser") to see in details

In [9]:
depart = (
    df.filter(
        pl.col("Event") == "離站",
        pl.col("Stop") == "新竹轉運站",
        pl.col("Direction") == "返程",
    )
    .with_columns(pl.col("Time").alias("DepartureTime"))
    .sort("Time")
)
depart

Route,Plate,Direction,Stop,Event,Time,DepartureTime
str,str,cat,str,cat,datetime[μs],datetime[μs]
"""17280""","""KKB-2058""","""返程""","""新竹轉運站""","""離站""",2026-02-25 05:15:46,2026-02-25 05:15:46
"""17280""","""KKB-2669""","""返程""","""新竹轉運站""","""離站""",2026-02-25 05:40:54,2026-02-25 05:40:54
"""17280""","""KKB-1715""","""返程""","""新竹轉運站""","""離站""",2026-02-25 06:02:19,2026-02-25 06:02:19
"""17280""","""KKB-2057""","""返程""","""新竹轉運站""","""離站""",2026-02-25 06:20:04,2026-02-25 06:20:04
"""17280""","""057-FS""","""返程""","""新竹轉運站""","""離站""",2026-02-25 06:42:05,2026-02-25 06:42:05
"""17280""","""KKB-2035""","""返程""","""新竹轉運站""","""離站""",2026-02-25 07:01:33,2026-02-25 07:01:33
"""17280""","""KKB-2055""","""返程""","""新竹轉運站""","""離站""",2026-02-25 08:01:17,2026-02-25 08:01:17
"""17280""","""KKB-2059""","""返程""","""新竹轉運站""","""離站""",2026-02-25 09:01:20,2026-02-25 09:01:20
"""17280""","""KKB-2670""","""返程""","""新竹轉運站""","""離站""",2026-02-25 10:03:37,2026-02-25 10:03:37


In [10]:
destination = (
    df.filter(
        pl.col("Event") == "進站",
        pl.col("Stop") == "龍潭運動公園",
        pl.col("Direction") == "返程",
    )
    .with_columns(pl.col("Time").alias("ArrivalTime"))
    .sort("Time")
)
destination

Route,Plate,Direction,Stop,Event,Time,ArrivalTime
str,str,cat,str,cat,datetime[μs],datetime[μs]
"""17280""","""KKB-2058""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 06:00:09,2026-02-25 06:00:09
"""17280""","""KKB-2669""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 06:23:58,2026-02-25 06:23:58
"""17280""","""KKB-1715""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 06:49:51,2026-02-25 06:49:51
"""17280""","""KKB-2057""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 07:02:17,2026-02-25 07:02:17
"""17280""","""057-FS""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 07:35:25,2026-02-25 07:35:25
"""17280""","""KKB-2035""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 07:50:22,2026-02-25 07:50:22
"""17280""","""KKB-2055""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 09:12:24,2026-02-25 09:12:24
"""17280""","""KKB-2059""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 10:08:11,2026-02-25 10:08:11
"""17280""","""KKB-2670""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 10:59:46,2026-02-25 10:59:46


## Finding
- Observation: There are 22 trips that depart from "新竹轉運站", among which 21 trips are recorded to pass through "龍潭運動公園" (a middle stop)
- Explanation: This suggests that if there is **no** passengers getting on/off at the stop, the record is not kept.
- Problem: Thee dataframes need some join logic to join properly in order to calculate the correct travel time

In [40]:
# check if the above finding the true for all the dates
min_date = date(2026, 1, 1)
max_date = date(2026, 2, 25)
diff = (max_date - min_date).days
dates = [min_date + timedelta(days=x) for x in range(diff)]

df = (
    df_full.filter(pl.col("SubRouteID").str.starts_with("1728"))
    .pipe(clean_df)
    .filter(pl.col("Time").is_between(min_date, max_date))
    .sort("Time")
    .collect(engine="streaming")
)

for d in dates:
    temp = df.filter(
        pl.col("Time").is_between(d, datetime(d.year, d.month, d.day, 23, 59, 59))
    )

    num_departure = temp.filter(
        pl.col("Event") == "離站",
        pl.col("Stop") == "新竹轉運站",
        pl.col("Direction") == "返程",
    ).shape[0]

    num_destination = temp.filter(
        pl.col("Event") == "進站",
        pl.col("Stop") == "龍潭運動公園",
        pl.col("Direction") == "返程",
    ).shape[0]

    if num_departure < num_destination:
        print(d)
    # else:
    #     print(num_departure)


2026-01-01
2026-01-07
2026-02-08


Turns out 3 out of the testing range has fewer rows for departing trips than arrival trips. Investigate more below.

In [47]:
temp = df.filter(
    pl.col("Time").is_between(date(2026, 1, 1), datetime(2026, 1, 1, 23, 59, 59))
)

temp_depart = temp.filter(
    pl.col("Event") == "離站",
    pl.col("Stop") == "新竹轉運站",
    pl.col("Direction") == "返程",
)

temp_dest = temp.filter(
    pl.col("Event") == "進站",
    pl.col("Stop") == "龍潭運動公園",
    pl.col("Direction") == "返程",
)

display(temp_depart)
display(temp_dest)

Route,Plate,Direction,Stop,StopSeq,Event,Time
str,str,cat,str,i32,cat,datetime[μs]
"""17280""","""057-FS""","""返程""","""新竹轉運站""",1,"""離站""",2026-01-01 06:01:45
"""17280""","""056-FS""","""返程""","""新竹轉運站""",1,"""離站""",2026-01-01 07:00:57
"""17280""","""KKA-1832""","""返程""","""新竹轉運站""",1,"""離站""",2026-01-01 10:04:33
"""17280""","""056-FS""","""返程""","""新竹轉運站""",1,"""離站""",2026-01-01 12:01:19
"""17280""","""KKB-1715""","""返程""","""新竹轉運站""",1,"""離站""",2026-01-01 13:00:43
"""17280""","""KKB-2057""","""返程""","""新竹轉運站""",1,"""離站""",2026-01-01 15:00:06
"""17280""","""KKB-2056""","""返程""","""新竹轉運站""",1,"""離站""",2026-01-01 16:03:52
"""17280""","""057-FS""","""返程""","""新竹轉運站""",1,"""離站""",2026-01-01 17:02:48
"""17280""","""058-FS""","""返程""","""新竹轉運站""",1,"""離站""",2026-01-01 18:01:29


Route,Plate,Direction,Stop,StopSeq,Event,Time
str,str,cat,str,i32,cat,datetime[μs]
"""17280""","""057-FS""","""返程""","""龍潭運動公園""",9,"""進站""",2026-01-01 06:54:56
"""17280""","""058-FS""","""返程""","""龍潭運動公園""",9,"""進站""",2026-01-01 08:56:05
"""17280""","""KKA-1832""","""返程""","""龍潭運動公園""",9,"""進站""",2026-01-01 10:56:17
"""17280""","""057-FS""","""返程""","""龍潭運動公園""",9,"""進站""",2026-01-01 12:08:23
"""17280""","""056-FS""","""返程""","""龍潭運動公園""",9,"""進站""",2026-01-01 12:53:47
"""17280""","""KKB-2057""","""返程""","""龍潭運動公園""",9,"""進站""",2026-01-01 15:56:15
"""17280""","""KKB-2056""","""返程""","""龍潭運動公園""",9,"""進站""",2026-01-01 17:02:50
"""17280""","""057-FS""","""返程""","""龍潭運動公園""",9,"""進站""",2026-01-01 18:17:37
"""17280""","""058-FS""","""返程""","""龍潭運動公園""",9,"""進站""",2026-01-01 19:02:19


In [46]:
temp_depart.join_asof(
    temp_dest, on="Time", by="Plate", strategy="forward", tolerance="2h"
).drop_nulls()

C:\Users\hank8\AppData\Local\Temp\ipykernel_5412\133068953.py:1: UserWarning:

Sortedness of columns cannot be checked when 'by' groups provided



Route,Plate,Direction,Stop,StopSeq,Event,Time,Route_right,Direction_right,Stop_right,StopSeq_right,Event_right
str,str,cat,str,i32,cat,datetime[μs],str,cat,str,i32,cat
"""17280""","""057-FS""","""返程""","""新竹轉運站""",1,"""離站""",2026-01-01 06:01:45,"""17280""","""返程""","""龍潭運動公園""",9,"""進站"""
"""17280""","""KKA-1832""","""返程""","""新竹轉運站""",1,"""離站""",2026-01-01 10:04:33,"""17280""","""返程""","""龍潭運動公園""",9,"""進站"""
"""17280""","""056-FS""","""返程""","""新竹轉運站""",1,"""離站""",2026-01-01 12:01:19,"""17280""","""返程""","""龍潭運動公園""",9,"""進站"""
"""17280""","""KKB-2057""","""返程""","""新竹轉運站""",1,"""離站""",2026-01-01 15:00:06,"""17280""","""返程""","""龍潭運動公園""",9,"""進站"""
"""17280""","""KKB-2056""","""返程""","""新竹轉運站""",1,"""離站""",2026-01-01 16:03:52,"""17280""","""返程""","""龍潭運動公園""",9,"""進站"""
"""17280""","""057-FS""","""返程""","""新竹轉運站""",1,"""離站""",2026-01-01 17:02:48,"""17280""","""返程""","""龍潭運動公園""",9,"""進站"""
"""17280""","""058-FS""","""返程""","""新竹轉運站""",1,"""離站""",2026-01-01 18:01:29,"""17280""","""返程""","""龍潭運動公園""",9,"""進站"""
"""17280""","""KKA-1832""","""返程""","""新竹轉運站""",1,"""離站""",2026-01-01 21:03:07,"""17280""","""返程""","""龍潭運動公園""",9,"""進站"""
"""17280""","""KKB-2056""","""返程""","""新竹轉運站""",1,"""離站""",2026-01-01 22:01:38,"""17280""","""返程""","""龍潭運動公園""",9,"""進站"""


In [12]:
result = depart.join_asof(
    destination,
    on="Time",
    by="Plate",
    strategy="forward",
    check_sortedness=False,
    tolerance="2h",
)
result = result.with_columns(
    ((pl.col("ArrivalTime") - pl.col("DepartureTime")).dt.total_seconds() / 60).alias(
        "Duration_min"
    )
)
result

Route,Plate,Direction,Stop,Event,Time,DepartureTime,Route_right,Direction_right,Stop_right,Event_right,ArrivalTime,Duration_min
str,str,cat,str,cat,datetime[μs],datetime[μs],str,cat,str,cat,datetime[μs],f64
"""17280""","""KKB-2058""","""返程""","""新竹轉運站""","""離站""",2026-02-25 05:15:46,2026-02-25 05:15:46,"""17280""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 06:00:09,44.383333
"""17280""","""KKB-2669""","""返程""","""新竹轉運站""","""離站""",2026-02-25 05:40:54,2026-02-25 05:40:54,"""17280""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 06:23:58,43.066667
"""17280""","""KKB-1715""","""返程""","""新竹轉運站""","""離站""",2026-02-25 06:02:19,2026-02-25 06:02:19,"""17280""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 06:49:51,47.533333
"""17280""","""KKB-2057""","""返程""","""新竹轉運站""","""離站""",2026-02-25 06:20:04,2026-02-25 06:20:04,"""17280""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 07:02:17,42.216667
"""17280""","""057-FS""","""返程""","""新竹轉運站""","""離站""",2026-02-25 06:42:05,2026-02-25 06:42:05,"""17280""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 07:35:25,53.333333
"""17280""","""KKB-2035""","""返程""","""新竹轉運站""","""離站""",2026-02-25 07:01:33,2026-02-25 07:01:33,"""17280""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 07:50:22,48.816667
"""17280""","""KKB-2055""","""返程""","""新竹轉運站""","""離站""",2026-02-25 08:01:17,2026-02-25 08:01:17,"""17280""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 09:12:24,71.116667
"""17280""","""KKB-2059""","""返程""","""新竹轉運站""","""離站""",2026-02-25 09:01:20,2026-02-25 09:01:20,"""17280""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 10:08:11,66.85
"""17280""","""KKB-2670""","""返程""","""新竹轉運站""","""離站""",2026-02-25 10:03:37,2026-02-25 10:03:37,"""17280""","""返程""","""龍潭運動公園""","""進站""",2026-02-25 10:59:46,56.15


In [13]:
px.bar(result, x="Time", y="Duration_min")